# Multilingual Health QA — End-to-End NLLB-200 & Hybrid RAG Pipeline

**Challenge:** Zindi Multilingual Health Question Answering in Low-Resource African Languages

**Repository:** [SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages](https://github.com/SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages.git)

### Complete Goals & Features Implemented:
1. **Generation Model (`facebook/nllb-200-distilled-600M`)**: Native FLORES-200 language code tokens (`aka_Latn`, `amh_Ethi`, `lug_Latn`, `swh_Latn`, `eng_Latn`) with per-language mini-batch grouping.
2. **Sentence-Embedding & Hybrid RAG Retrieval**: Blends lexical TF-IDF subword n-grams and multilingual dense embeddings (`sentence-transformers/paraphrase-multilingual-mpnet-base-v2`) in `src/retrieval.py`.
3. **Tightened Threshold Optimization**: Fine-grained `0.01` grid search and Logistic Regression score calibrator in `src/threshold_optimizer.py`.
4. **Per-Language Epoch ROUGE Callback**: Live per-subset ROUGE-1 & ROUGE-L table breakdown during fine-tuning.
5. **TargetLLM Post-Processing**: Automated special token cleaning and submission verification (`ID`, `TargetRLF1`, `TargetR1F1`, `TargetLLM`).

In [ ]:
# 1. Environment & Repository Setup (Universal Home Root Reset)
import os, sys, shutil, zipfile, urllib.request
from pathlib import Path

# Reset working directory to home root to avoid stale CWD or nested directories
try:
    home_dir = Path.home()
    if (Path('/home/jovyan')).exists():
        home_dir = Path('/home/jovyan')
    elif (Path('/content')).exists():
        home_dir = Path('/content')
    os.chdir(home_dir)
except Exception as e:
    print(f'[WARN] Directory reset fallback: {e}')

repo_name = 'Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages'
zip_url = 'https://github.com/SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages/archive/refs/heads/main.zip'

# Clean up any previous folder to ensure fresh code and datasets
if os.path.exists(repo_name):
    print(f"Cleaning previous '{repo_name}' directory...")
    shutil.rmtree(repo_name, ignore_errors=True)

if os.path.exists('repo.zip'):
    os.remove('repo.zip')

print('Downloading repository & raw datasets from GitHub...')
urllib.request.urlretrieve(zip_url, 'repo.zip')

print('Extracting project files...')
with zipfile.ZipFile('repo.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

if os.path.exists(f'{repo_name}-main'):
    os.rename(f'{repo_name}-main', repo_name)

%cd {repo_name}
print('\nSetup complete! Project root:')
!pwd


In [ ]:
# 2. Install Required Dependencies
!pip install -q torch transformers datasets evaluate scikit-learn pandas numpy rouge-score sentence-transformers accelerate peft
print('All package dependencies installed successfully!')


In [ ]:
# 3. Initialize Master Paths, Hardware Diagnostics & Project Imports
import os, sys, re, json, subprocess
from pathlib import Path

BASE_DIR = Path.cwd()
for path_to_add in [str(BASE_DIR), str(BASE_DIR / 'src')]:
    if path_to_add not in sys.path:
        sys.path.insert(0, path_to_add)

DATA_DIR        = BASE_DIR / 'data' / 'raw'
SUBMISSIONS_DIR = BASE_DIR / 'submissions'
CHECKPOINTS_DIR = BASE_DIR / 'models' / 'checkpoints'
SRC_PATH        = BASE_DIR / 'src' / 'nllb_pipeline.py'

SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'Training set.csv'
VAL_PATH   = DATA_DIR / 'Validation set.csv'
TEST_PATH  = DATA_DIR / 'Test set.csv'

import torch
import pandas as pd
import numpy as np
from retrieval import HybridRetriever
from threshold_optimizer import optimize_per_subset_thresholds

print(f'BASE_DIR: {BASE_DIR}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Capacity: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

print('\nDataset Verification:')
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH, SRC_PATH]:
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}] {p.relative_to(BASE_DIR)}')


In [ ]:
# 4. Load Data & Preview Language Subsets
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f'Train shape : {train_df.shape}')
print(f'Val shape   : {val_df.shape}')
print(f'Test shape  : {test_df.shape}')

print('\nSubset distribution in train data:')
display(train_df['subset'].value_counts())


In [ ]:
# 5. Interactive Test of Hybrid Dense + Lexical RAG Retriever & Threshold Grid
print('Initializing Hybrid RAG Retriever (TF-IDF + Dense Multilingual Embeddings)...')
hybrid_retriever = HybridRetriever(train_df, enable_dense=True)

val_retr_preds, val_retr_sims = [], []
for _, row in val_df.iterrows():
    ans, sim = hybrid_retriever.get_top1(row['input'], row['subset'], exclude_exact=True)
    val_retr_preds.append(ans)
    val_retr_sims.append(sim)

print(f'Retrieved {len(val_retr_preds)} reference answers for validation questions.')


In [ ]:
# 6. Fast Dry-Run Verification (Testing Pipeline End-to-End)
dry_run_cmd = [
    sys.executable, str(SRC_PATH),
    '--dry_run',
    '--model_name', 'facebook/nllb-200-distilled-600M',
    '--submission_path', str(SUBMISSIONS_DIR / 'submission_dry_run.csv'),
]

print('Running fast dry-run verification with live log streaming:')
process = subprocess.Popen(
    dry_run_cmd, cwd=str(BASE_DIR),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
assert process.returncode == 0, f'Dry run failed with code {process.returncode}'


In [ ]:
# 7. Execute Full NLLB-200 Fine-Tuning & Hybrid RAG Production Pipeline
full_run_cmd = [
    sys.executable, str(SRC_PATH),
    '--model_name', 'facebook/nllb-200-distilled-600M',
    '--use_rag',
    '--use_dense_rag',
    '--epochs', '3',
    '--batch_size', '8',
    '--learning_rate', '5e-5',
    '--train_path', str(TRAIN_PATH),
    '--val_path', str(VAL_PATH),
    '--test_path', str(TEST_PATH),
    '--output_dir', str(CHECKPOINTS_DIR / 'nllb-health-qa-checkpoint'),
    '--submission_path', str(SUBMISSIONS_DIR / 'submission_nllb_hybrid.csv'),
]

print('🚀 Launching Full Production NLLB-200 Fine-Tuning & Hybrid RAG Pipeline...')
process = subprocess.Popen(
    full_run_cmd, cwd=str(BASE_DIR),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
if process.returncode == 0:
    print('\n🎉 Full production pipeline execution finished successfully!')
else:
    print(f'\n❌ Process exited with return code {process.returncode}')


In [ ]:
# 8. Inspect & Validate Zindi Submission CSV Output
sub_path = SUBMISSIONS_DIR / 'submission_nllb_hybrid.csv'
if sub_path.exists():
    sub_df = pd.read_csv(sub_path)
    print(f'✅ Submission shape      : {sub_df.shape}')
    print(f'✅ Expected shape        : ({len(test_df)}, 4)')
    print(f'✅ Required columns      : {sub_df.columns.tolist()}')
    print(f'✅ Total missing values  : {sub_df.isna().sum().sum()}')
    print(f'✅ TargetRLF1 == TargetLLM: {(sub_df["TargetRLF1"] == sub_df["TargetLLM"]).all()}')
    print('\nTop 10 Submission Predictions:')
    display(sub_df.head(10))
else:
    print(f'Submission file not found at {sub_path}')


In [ ]:
# 9. Optional File Download (Google Colab)
try:
    from google.colab import files
    files.download(str(SUBMISSIONS_DIR / 'submission_nllb_hybrid.csv'))
except ImportError:
    print(f'Local submission saved at: {SUBMISSIONS_DIR / "submission_nllb_hybrid.csv"}')
